# ATiG 2026: LT-FH exercise using ltpred

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bvilhjal/ATIG_2026/blob/main/teaching_days/2026-09-24/LTFH/LTFH_exercise.ipynb)

## What you will learn

You score a simulated register with [ltpred](https://github.com/bvilhjal/ltpred)'s
family-history method, LT-FH++, and compare the scores with the true genetic liability,
which a simulation lets you see. You make eight plots along the way, starting with
survival curves that show what is in the data before any model is fitted.

- **In class (about an hour):** Part 0 (15 min) inspects the register and scores it; **A** (20 min):
  what a wrong heritability h² does; **B** (30 min): how well the score predicts who is
  diagnosed later.
- **Homework:** **C** the score against a yes/no family history; **D** heritability from
  parents and children; **E** the two scales on which h² is reported.

Work in pairs, ideally one person with a biology and one with a bioinformatics
background. Questions marked **Discuss** are for talking through together.

**Background.** LT-FH ([Hujoel et al. 2020](https://doi.org/10.1038/s41588-020-0613-6))
turns relatives' diagnoses into an estimate of a person's genetic liability; LT-FH++
([Pedersen et al. 2022](https://doi.org/10.1016/j.ajhg.2022.01.009)) adds age of onset
through age-dependent thresholds. ltpred computes it with the Pearson–Aitken method of
PA-FGRS ([Krebs et al. 2024](https://doi.org/10.1016/j.ajhg.2024.09.009)).

## The idea in five sentences

1. Everyone has a **liability** to the disease: an unobserved quantity that adds up a
   genetic part and an environmental part and follows a bell curve in the population.
2. People whose liability passes a **threshold** get the disease. In LT-FH++ the threshold
   falls with age and can differ by sex, so the higher your liability, the earlier you are
   diagnosed.
3. **Heritability h²** is the share of liability variance that is genetic. Relatives share
   genes, so their diagnoses carry information about your genetic liability.
4. **LT-FH++** turns the relatives' diagnoses and ages into a best estimate of your genetic
   liability (the *score*), with a measure of its uncertainty.
5. Real data never reveal the true genetic liability. A simulation does, so here you can
   check how good the score is.

| Term | Meaning here |
|---|---|
| liability | the unobserved disease predisposition; genetic part `g` plus the rest |
| h² (liability scale) | share of liability variance due to additive genes; 0.5 here |
| K, prevalence | fraction who get the disease in a lifetime: 12% of men, 8% of women, 10% overall |
| cumulative incidence (CIP) | fraction diagnosed by a given age; rises from 0 to K |
| Kaplan–Meier | the standard estimate of cumulative incidence from follow-up data |
| score, posterior mean | best estimate of `g` given the family's records |
| AUC | chance that a random case scores higher than a random non-case (0.5 = coin flip) |
| calibration slope | slope of the truth regressed on the score; 1 means the score's scale is right |
| tetrachoric correlation | correlation of two liabilities, inferred from two yes/no variables |
| observed scale | h² measured on the 0/1 diagnosis itself instead of on the liability |

## How to use this notebook

1. Click **Open in Colab** above, then **File → Save a copy in Drive** and work in your copy.
2. Run the cells in order with **Shift+Enter**. If Colab restarts, start again from the top.
3. Where a cell has `...`, replace it with one expression; the comment says what. Every
   plot is ready to run once its `...` is filled. **Checkpoints** say what you should see.

| Name | What it is |
|---|---|
| `reg` | the simulated register (one row per person) |
| `male`, `sex` | each person's sex (True/False, and "M"/"F") |
| `g` | each person's true genetic liability |
| `est` | everyone's score, using all records |
| `free40` | True for people still undiagnosed at 40 |
| `est40`, `var40` | the score as known at 40, and its uncertainty, for those people |
| `score(h2)` | rescores everyone under an assumed h² |

### The Python you need

| Code | What it does |
|---|---|
| `x[mask]` | keeps the entries of `x` where the True/False array `mask` is True |
| `a & b`, `~a` | True where both are True; True where `a` is False |
| `x.mean()`, `x.sum()` | average and total; on True/False arrays, the share and count of True |
| `[f(x) for x in xs]` | a list with `f` applied to each element of `xs` |
| `ax.step(x, y)`, `ax.scatter(x, y)`, `ax.hist(x)`, `ax.bar(names, values)`, `ax.plot(x, y)` | the plots you will make |

## Setup

The first cell installs ltpred; the second loads it, loads matplotlib for plotting and
defines two small helpers, `corr` and `auc`. (On your own computer instead of Colab? See
the [setup page](https://bvilhjal.github.io/ATIG_2026/setup.html).)

In [ ]:
%pip install -q git+https://github.com/bvilhjal/ltpred@v0.7.1     # installs ltpred (about 30 s)

In [1]:
import numpy as np
from scipy.stats import norm, rankdata
import matplotlib.pyplot as plt

import ltpred
from ltpred import (simulate_pedigree, simulate_register_liabilities, estimate_liabilities,
                    kaplan_meier_cip)
from dataclasses import replace
print("ltpred", ltpred.__version__)


def corr(x, y):
    """Pearson correlation of two arrays."""
    return np.corrcoef(x, y)[0, 1]


def auc(score, case):
    """AUC: the chance that a random case scores above a random non-case.

    Computed from ranks (the Mann-Whitney statistic); `case` is a boolean array."""
    r = rankdata(score)
    n1, n0 = case.sum(), (~case).sum()
    return (r[case].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

ltpred 0.7.1


## Part 0. A register where the truth is known

The cell below simulates three generations of families (5,075 people). Each person gets
a liability: a genetic part `g` with variance h² = 0.5 plus a non-genetic part. A person
is diagnosed at the age their liability crosses an age- and sex-dependent threshold (the
LT-FH++ model): the disease is more common, and starts earlier, in men. Everyone is
followed to age 70. Fathers are men and mothers women; people who never became parents
get a random sex. The cell simulates the liabilities once and reads each person's
records off their own sex's incidence curve; run it as is.

| Field | Meaning |
|---|---|
| `reg.ids`, `reg.father`, `reg.mother` | the pedigree |
| `reg.status` | diagnosed by age 70 (True/False) |
| `male` | True for men |
| `reg.age` | age at diagnosis, or 70 for the undiagnosed |
| `reg.onset`, `reg.birth_time` | age at onset (`inf` if never) and calendar time of birth |
| `reg.genetic` | the true genetic liability, stored as `g` |

In [2]:
H2 = 0.5                                        # true liability-scale heritability
AGES = np.arange(0, 121.0)                      # ages 0, 1, ..., 120
CIP_M = 0.12 / (1 + np.exp((58 - AGES) / 8))    # men: lifetime 12%, half of it by age 58
CIP_F = 0.08 / (1 + np.exp((62 - AGES) / 8))    # women: lifetime 8%, half of it by age 62
K = 0.10                                        # lifetime prevalence, men and women together

ids, father, mother = simulate_pedigree(np.random.default_rng(1), n_founder_pairs=500, gens=2)
fathers, mothers = set(father), set(mother)
coin = np.random.default_rng(2).random(len(ids)) < 0.5  # a random sex for people who never became parents
male = np.array([p in fathers or (p not in mothers and c) for p, c in zip(ids, coin)])
sex = np.where(male, "M", "F")


def simulate(cip):
    """The register with one incidence curve for everyone (same seed = same liabilities)."""
    return simulate_register_liabilities(np.random.default_rng(1), ids, father, mother,
                                         h2=H2, cip_ages=AGES, cip_values=cip, eval_age=70)


as_men, as_women = simulate(CIP_M), simulate(CIP_F)
# each person's records follow their own sex's curve: sex-specific thresholds, as in LT-FH++
reg = replace(as_men, status=np.where(male, as_men.status, as_women.status),
              age=np.where(male, as_men.age, as_women.age),
              onset=np.where(male, as_men.onset, as_women.onset))
g = reg.genetic                                 # the truth, known only because we simulated it

print(f"{len(reg.ids)} people ({male.sum()} men), {reg.status.sum()} diagnosed by age 70")
print(f"diagnosed: men {reg.status[male].mean():.1%}, women {reg.status[~male].mean():.1%}")

5075 people (2482 men), 366 diagnosed by age 70
diagnosed: men 9.0%, women 5.5%


**Checkpoint.** 5,075 people (2,482 men), 366 diagnosed: 9.0% of men and 5.5% of women.

### Look at the data first

Before any model, inspect what the register holds. **Cumulative incidence** is the share
diagnosed by each age; one minus it is the *survival* curve of disease-free people. With
follow-up that ends early for some people, the standard estimate is **Kaplan–Meier**,
and ltpred has it: `kaplan_meier_cip(entry ages, exit ages, events)`. Here everyone enters
at birth (age 0) and leaves at diagnosis or at 70.

### Q1: Plot the cumulative incidence by sex.

Fill in the exit ages and the events for the people in `keep`. Do the observed curves
follow the true (dashed) curves? Why would one curve for everyone be a problem?

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
for label, keep, cip, colour in (("men", male, CIP_M, "C0"), ("women", ~male, CIP_F, "C1")):
    # Kaplan-Meier: everyone enters at birth and leaves at diagnosis or at 70
    km = kaplan_meier_cip(np.zeros(keep.sum()), ..., ...)    # exit ages and events: reg.age[keep], reg.status[keep]
    ax.step(km.ages, 100 * km.values, where="post", color=colour, label=f"{label}: observed")
    ax.plot(AGES[:71], 100 * cip[:71], ls="--", color=colour, label=f"{label}: true curve")
ax.set_xlabel("age")
ax.set_ylabel("% diagnosed by this age")
ax.legend()
plt.show()

<details><summary><b>Hint</b></summary>

The comment names them: `reg.age[keep]` (age at diagnosis, or 70) and `reg.status[keep]` (True if diagnosed).

</details>

**Checkpoint.** Two rising step curves, the men's above the women's, each close to its dashed curve.

### Q2: Plot the cumulative incidence for people with and without a diagnosed parent.

The cell finds who has a diagnosed parent. Fill in the comparison group: people whose
parents are in the register but neither was diagnosed. What do the curves tell you before
any model is fitted?

In [ ]:
row = {p: i for i, p in enumerate(reg.ids)}                  # id -> row number
has_parents = np.array([f in row for f in reg.father])       # parents recorded in the register
parent_dx = np.array([any(reg.status[row[p]] for p in (f, m) if p in row)
                      for f, m in zip(reg.father, reg.mother)])   # a parent diagnosed by 70

fig, ax = plt.subplots(figsize=(6.5, 4))
for label, keep in (("a diagnosed parent", parent_dx),
                    ("no diagnosed parent", ...)):          # parents recorded, but neither diagnosed
    km = kaplan_meier_cip(np.zeros(keep.sum()), reg.age[keep], reg.status[keep])
    ax.step(km.ages, 100 * km.values, where="post", label=f"{label} (n = {keep.sum()})")
ax.set_xlabel("age")
ax.set_ylabel("% diagnosed by this age")
ax.legend()
plt.show()

<details><summary><b>Hint</b></summary>

People with recorded parents are `has_parents`; neither diagnosed is `~parent_dx`. Combine them with `&`.

</details>

**Checkpoint.** The curve for a diagnosed parent rises clearly above the other.

### The scorer in one call

`estimate_liabilities` takes the pedigree, everyone's diagnosis and age, the incidence
curves and h², and returns each person's score (`.est`) with its uncertainty (`.var`).
`strata=sex` with `cip_by_stratum` gives each sex its own curve, as Q1 showed it needs.
The comments explain each argument.

In [3]:
curves = {"M": (AGES, CIP_M, 0.12), "F": (AGES, CIP_F, 0.08)}   # sex -> (ages, curve, lifetime K)

scores = estimate_liabilities(
    reg.ids, reg.father, reg.mother,          # the pedigree: who is whose parent
    probands=reg.ids,                         # whom to score: everyone
    status=reg.status, age=reg.age,           # diagnosed by 70? age at diagnosis or at 70
    strata=sex, cip_by_stratum=curves,        # each sex has its own incidence curve, so its own thresholds
    h2=H2,                                    # heritability: you supply it, the scorer never estimates it
    use="gwas")                               # also use each person's own diagnosis

est, var = scores.est, scores.var
print(f"corr(score, g) = {corr(est, g):.3f}")
print(f"var(score) = {est.var():.3f}  +  mean posterior variance = {var.mean():.3f}"
      f"  =  {est.var() + var.mean():.3f}")

corr(score, g) = 0.529
var(score) = 0.128  +  mean posterior variance = 0.359  =  0.488


### The helpers used below

`score(h2)` repeats that call under any assumed h². `score_at_40(h2)` scores the people
still undiagnosed at 40 **as of their 40th birthday**, hiding their own status and every
later record: the honest setting for predicting who is diagnosed later.

In [4]:
free40 = reg.onset > 40                     # still undiagnosed on their 40th birthday


def score(h2):
    """Everyone's score from all records up to 70 (use="gwas"): (mean, variance)."""
    s = estimate_liabilities(reg.ids, reg.father, reg.mother, probands=reg.ids,
                             status=reg.status, age=reg.age, strata=sex, cip_by_stratum=curves,
                             h2=h2, use="gwas")
    return s.est, s.var


def score_at_40(h2):
    """Scores as known on each person's 40th birthday (use="prediction"), for the
    people still undiagnosed then: their own status and every later record are hidden."""
    s = estimate_liabilities(reg.ids, reg.father, reg.mother,
                             probands=[p for p, keep in zip(reg.ids, free40) if keep],
                             status=reg.status, age=reg.age, strata=sex, cip_by_stratum=curves,
                             h2=h2, use="prediction",
                             birth_time=reg.birth_time,                  # everyone's birth date
                             index_time=(reg.birth_time + 40)[free40])   # each proband's 40th birthday
    return s.est, s.var


est40, var40 = score_at_40(H2)
print(f"{free40.sum()} people undiagnosed at 40; corr(score at 40, g) = {corr(est40, g[free40]):.3f}")

5040 people undiagnosed at 40; corr(score at 40, g) = 0.280


### Q3: Plot the score against the true genetic liability.

Fill in the two arrays: the score `est` on the x-axis and the truth `g` on the y-axis.
What does a correlation of about 0.5 look like?

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(..., ..., s=3, alpha=0.4)           # x: the score est, y: the truth g
ax.set_xlabel("score (posterior mean)")
ax.set_ylabel("true genetic liability g")
ax.set_title(f"corr = {corr(est, g):.2f}")
plt.show()

<details><summary><b>Hint</b></summary>

`ax.scatter(x, y)`: here `x` is `est` and `y` is `g`.

</details>

**Checkpoint.** A wide cloud tilted upwards, with the correlation 0.53 in the title.

## Part A. What does a wrong h² do?

In a real analysis h² comes from outside (a twin, pedigree or SNP study) and you supply
it. Here you can pass a wrong value on purpose and compare with the truth.

### Q4: If you tell the scorer h² = 0.8 when the truth is 0.5, do the scores spread more or less? Does their ranking change?

Write down your expectation with one sentence of reasoning. Hint: h² is the share of
liability variance that is genetic, and therefore shared with relatives.

**Discuss** with your partner.

### Q5: Plot the truth against the score for h² = 0.2, 0.5 and 0.8.

Each panel shows the fitted line of `g` on the score (orange) and the dashed diagonal.
Fill in the slope. A score is **calibrated** when the slope is 1: among people scored 0.3,
the true liability averages 0.3. Which assumed h² gives slope 1, and what happens to the
correlation?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharex=True, sharey=True)
for ax, h2 in zip(axes, (0.2, 0.5, 0.8)):
    e, v = score(h2)                           # rescore everyone under this h2
    b = ...                                    # slope of g regressed on e: np.polyfit(e, g, 1)[0]
    ax.scatter(e, g, s=2, alpha=0.3)
    ax.axline((0, 0), slope=1, ls="--", color="grey")   # slope 1: calibrated
    ax.axline((0, 0), slope=b, color="C1")              # the fitted slope
    ax.set_title(f"assumed h² = {h2}\ncorr {corr(e, g):.3f}, slope {b:.2f}")
    ax.set_xlabel("score")
axes[0].set_ylabel("true g")
plt.show()

<details><summary><b>Hint</b></summary>

The comment gives the expression: `np.polyfit(e, g, 1)[0]` is the slope of `g` regressed on `e`.

</details>

**Checkpoint.** The correlations in the three titles are almost equal; the slopes are
not.

## Part B. Who is diagnosed between 40 and 70?

![Liability thresholds at 40 and 70. Higher liability is earlier onset. People still undiagnosed at 40 are left of T40; those diagnosed by 70 have crossed T70.](https://bvilhjal.github.io/ATIG_2026/teaching_days/2026-09-24/LTFH/figures/thresholds.png)

A correlation with `g` is not the accuracy of predicting a later diagnosis. Here you use
the score at 40 (`est40`) and what happened next.

### Q6: How many of the people undiagnosed at 40 are diagnosed by 70?

`reg.status` means "diagnosed by 70" and `free40` means "undiagnosed at 40". The result
`y` must line up with `est40`, which has one entry per person in `free40`.

In [ ]:
y = ...          # one boolean per person in est40: diagnosed between 40 and 70
print(f"{y.sum()} incident cases among {len(y)} people ({y.mean():.1%})")

<details><summary><b>Hint</b></summary>

Keep the entries of `reg.status` for the people in `free40` (the `x[mask]` pattern). Everyone in `free40` was undiagnosed at 40, so diagnosed by 70 means diagnosed between 40 and 70.

</details>

**Checkpoint.** A few hundred cases among 5,040 people.

### Q7: Plot the scores of the later cases and the non-cases, and compute the AUC.

Fill in the cases' scores for the histogram and `g` for the same people in the AUC.
How much do the two histograms overlap? Why is even the true `g` below AUC 1, and why is
the `use="gwas"` score (which saw each person's own diagnosis) near-perfect?

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(est40[~y], bins=40, density=True, alpha=0.5, label="not diagnosed")
ax.hist(..., bins=40, density=True, alpha=0.5, label="diagnosed 40 to 70")   # the cases' scores
ax.set_xlabel("score at 40")
ax.legend()
plt.show()

print(f"AUC score at 40        {auc(est40, y):.3f}")
print(f"AUC true g             {auc(..., y):.3f}")      # g for the same people
print(f"AUC use='gwas' score   {auc(est[free40], y):.3f}")

<details><summary><b>Hint</b></summary>

The non-cases' histogram uses `est40[~y]`; the cases' uses `est40[y]`. For the AUC, select `g` for the same people with `[free40]`.

</details>

**Checkpoint.** The histograms overlap a lot; the AUCs are roughly 0.6, 0.9 and above
0.99.

### Q8: Turn the score into a risk and plot it against what happened.

Given the relatives, a person's liability is normal with mean `est40` and standard
deviation `sd`; the risk of diagnosis between 40 and 70 is the chance it lies between the
thresholds, given it is below the age-40 threshold. The cell computes that risk for you.
Fill in the mean predicted risk in each fifth of the score, then compare the bars. Does
the risk match the observed rates? What would a wrong h² do to it?

In [ ]:
cip40 = np.where(male, np.interp(40, AGES, CIP_M), np.interp(40, AGES, CIP_F))[free40]
cip70 = np.where(male, np.interp(70, AGES, CIP_M), np.interp(70, AGES, CIP_F))[free40]
T40, T70 = norm.isf(cip40), norm.isf(cip70)            # each person's LT-FH++ thresholds at 40 and 70
sd = np.sqrt(var40 + 1 - H2)                           # spread of full liability given the relatives
below40 = norm.cdf((T40 - est40) / sd)                 # P(undiagnosed at 40)
below70 = norm.cdf((T70 - est40) / sd)                 # P(undiagnosed at 70)
risk = (below40 - below70) / below40                   # P(diagnosed 40-70 | undiagnosed at 40)

fifths = np.array_split(np.argsort(est40, kind="stable"), 5)   # lowest to highest score
observed = [y[idx].mean() for idx in fifths]                   # share diagnosed in each fifth
predicted = [...]                                              # mean risk in each fifth

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(np.arange(5) - 0.2, observed, width=0.4, label="observed")
ax.bar(np.arange(5) + 0.2, predicted, width=0.4, label="predicted risk")
ax.set_xticks(range(5), ["lowest", "2nd", "middle", "4th", "highest"])
ax.set_xlabel("fifth of the score at 40")
ax.set_ylabel("share diagnosed 40 to 70")
ax.legend()
plt.show()
print(f"mean predicted risk {risk.mean():.3f}   observed {y.mean():.3f}")

<details><summary><b>Hint</b></summary>

Copy the `observed` line and replace `y[idx]` with `risk[idx]`.

</details>

**Checkpoint.** Both sets of bars rise from the lowest to the highest fifth, and the mean
predicted risk is within about half a percentage point of the observed rate.

## Part C. The score against a yes/no family history (homework)

A common shortcut is the indicator "any parent or full sibling diagnosed". The cell below
builds it twice: with follow-up to 70 (`fh`) and as known on each person's 40th birthday
(`fh40`). Run it as is.

In [5]:
row = {p: i for i, p in enumerate(reg.ids)}          # id -> row number
children = {}                                        # (father, mother) -> rows of their children
for i, (f, m) in enumerate(zip(reg.father, reg.mother)):
    if f in row and m in row:
        children.setdefault((f, m), []).append(i)


def first_degree(i):
    """Rows of person i's parents and full siblings."""
    f, m = reg.father[i], reg.mother[i]
    parents = [row[p] for p in (f, m) if p in row]
    siblings = [j for j in children.get((f, m), []) if j != i]
    return np.array(parents + siblings, dtype=int)


fdr = [first_degree(i) for i in range(len(reg.ids))]
diag_time = reg.birth_time + reg.onset               # calendar time of each diagnosis
birth40 = reg.birth_time + 40                        # calendar time of each 40th birthday

# yes/no family history: any affected parent or sibling, by 70 and by one's own 40th birthday
fh = np.array([reg.status[r].any() for r in fdr])
fh40 = np.array([(reg.status[r] & (diag_time[r] <= birth40[i])).any() for i, r in enumerate(fdr)])
print(f"family-history positive: {fh.sum()} by age 70, {fh40.sum()} at their 40th birthday")

family-history positive: 971 by age 70, 627 at their 40th birthday


### Q9: Plot how much of `g` each measure explains.

Fill in R², the squared correlation with `g`, for own status, the indicator and the
score. The last line compares the indicator known at 40 with the score at 40 as
predictors of Part B's outcome. Why is own status left out of that comparison?

In [ ]:
names = ["own status", "FH indicator", "score (use='gwas')"]
r2 = [... for x in (reg.status, fh, est)]      # R²: the squared correlation of each with g

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.bar(names, r2)
ax.set_ylabel("R² with the true g")
plt.show()
print(f"prediction at 40: AUC FH indicator {auc(fh40[free40], y):.3f}, AUC score {auc(est40, y):.3f}")

<details><summary><b>Hint</b></summary>

R² is the squared correlation: `corr(x, g) ** 2`.

</details>

**Checkpoint.** The score's bar is clearly the tallest.

## Part D. Where does h² come from? (homework)

A quick cross-check on h²: it is about twice the **tetrachoric correlation** between
parents' and children's diagnoses, the correlation of the underlying liabilities inferred
from a 2×2 table. The cell below lists every parent–child pair; run it as is.

In [6]:
from ltpred import tetrachoric

# every parent-child pair in the register, as row numbers
pairs = [(row[p], i) for i, (f, m) in enumerate(zip(reg.father, reg.mother))
         for p in (f, m) if p in row]
par, kid = np.array(pairs).T


def table(a, b):
    """2x2 table of two True/False arrays: both, first only, second only, neither."""
    return np.array([(a & b).sum(), (a & ~b).sum(), (~a & b).sum(), (~a & ~b).sum()])

### Q10: Estimate h² from the parent–offspring pairs.

In [ ]:
p_status, k_status = reg.status[par], reg.status[kid]
t = ...                      # tetrachoric(first status array, second status array)
print(f"{len(par)} pairs, table {table(p_status, k_status)}")
print(f"h2 = 2 rho = {2 * t.rho:.2f} +/- {2 * t.se:.2f}   (truth {H2})")

<details><summary><b>Hint</b></summary>

`tetrachoric` takes the two True/False arrays: the parents' statuses and the children's.

</details>

**Checkpoint.** An estimate within one standard error of 0.5.

### Q11: Why does twice the tetrachoric work here, and what would break it in a real register?

Think about the threshold, shared environment, and how the register was sampled.

**Discuss** with your partner.

## Part E. Which scale is h² on? (homework)

![A liability split into a 0/1 outcome. The right tail is the cases, 10% when K = 0.10.](https://bvilhjal.github.io/ATIG_2026/teaching_days/2026-09-24/LTFH/figures/observed_scale.png)

GWAS methods such as GREML and LD-score regression report h² on the observed 0/1 scale,
and that number depends on the sample's case proportion P
([Lee et al. 2011](https://doi.org/10.1016/j.ajhg.2011.02.002)). ltpred's
`liability_to_observed_h2(h2, K, P)` converts a liability-scale h² to the observed scale.

### Q12: Plot the observed-scale h² that studies with different case fractions would report.

Fill in the list. The truth is h² = 0.5 with K = 0.10. What would a population GWAS
report, and what would happen if you passed that number to the scorer unconverted?

In [ ]:
from ltpred import liability_to_observed_h2

Ps = np.linspace(0.02, 0.6, 50)                # case fraction in the sample
h2_obs = [...]                                 # observed-scale h² for each P: liability_to_observed_h2(H2, K, P)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(Ps, h2_obs)
ax.axhline(H2, ls="--", color="grey", label="liability scale: 0.5")
ax.axvline(K, ls=":", color="C1", label="population sample: P = K")
ax.set_xlabel("case fraction in the sample, P")
ax.set_ylabel("observed-scale h²")
ax.legend()
plt.show()
print(f"population sample {float(liability_to_observed_h2(H2, K, None)):.3f}, "
      f"1:1 study {float(liability_to_observed_h2(H2, K, 0.5)):.3f}")

<details><summary><b>Hint</b></summary>

`[float(liability_to_observed_h2(H2, K, P)) for P in Ps]`: one value per case fraction.

</details>

**Checkpoint.** A curve well below 0.5 at P = K that rises towards the middle; 0.171 for
a population sample.

### Q13: Analysis note.

In at most 100 words: what does the family-history score estimate, what did it show in
this exercise, and what can it not establish? Cite one number from each part you did.

**Discuss** with your partner.

---
Part of [ATIG 2026](https://github.com/bvilhjal/ATIG_2026), teaching day
[24 September](https://github.com/bvilhjal/ATIG_2026/tree/main/teaching_days/2026-09-24).